# 2.4 Gold

Builds the gold layer: a customer dimension and a One Big Table (OBT) that joins all facts with their dimensions, ready for direct BI / ad-hoc use.

**What this notebook covers:**
- Building `dim_customer` by joining silver customers with their pivoted addresses
- Creating `obt_orders` — a fully denormalized table where every line item already carries all customer and address attributes
- Example analytics queries

**OBT vs. star schema:**  
An OBT trades storage for query simplicity — no joins needed at query time. Works well for self-service analytics and BI tools. For very large or frequently-updated dimensions, a proper star schema is more maintainable.

**Inputs:**

| Source | Layer |
|--------|-------|
| `de_lab.silver.customers` | Silver |
| `de_lab.silver.addresses` | Silver |
| `de_lab.silver.fact_line_items` | Silver |

**Outputs:**

| Table | Description |
|-------|-------------|
| `de_lab.gold.dim_customer` | Customer dimension with shipping and billing addresses |
| `de_lab.gold.obt_orders` | Fully denormalized order line items for analytics |

## dim_customer

In [0]:
CREATE OR REPLACE TABLE de_lab.gold.dim_customer AS
SELECT
  c.customer_id,
  c.customer_name,
  c.email,
  c.date_of_birth,
  c.age,
  c.phone,
  c.created_timestamp,
  a.Shipping_address_line_1,
  a.Shipping_city,
  a.Shipping_state,
  a.Shipping_postcode,
  a.Billing_address_line_1,
  a.Billing_city,
  a.Billing_state,
  a.Billing_postcode
FROM de_lab.silver.customers c
LEFT JOIN de_lab.silver.addresses a ON c.customer_id = a.customer_id;

In [0]:
SELECT * FROM de_lab.gold.dim_customer LIMIT 10;

## obt_orders

In [0]:
CREATE OR REPLACE TABLE de_lab.gold.obt_orders AS
SELECT
  f.order_id,
  f.order_status,
  f.transaction_timestamp,
  f.order_date,
  f.total_amount,
  f.item_id,
  f.item_name,
  f.price,
  f.quantity,
  (f.price * f.quantity) AS line_total,
  f.category,
  c.customer_id,
  c.customer_name,
  c.email,
  c.date_of_birth,
  c.age,
  c.phone,
  c.Shipping_address_line_1,
  c.Shipping_city,
  c.Shipping_state,
  c.Shipping_postcode,
  c.Billing_address_line_1,
  c.Billing_city,
  c.Billing_state,
  c.Billing_postcode
FROM de_lab.silver.fact_line_items f
LEFT JOIN de_lab.gold.dim_customer c ON f.customer_id = c.customer_id;

In [0]:
SELECT * FROM de_lab.gold.obt_orders ORDER BY order_id LIMIT 10;

## Analytics

In [0]:
SELECT
  category,
  Shipping_city,
  Shipping_state,
  COUNT(DISTINCT order_id)    AS total_orders,
  SUM(quantity)               AS total_units_sold,
  ROUND(SUM(total_amount), 2) AS total_revenue
FROM de_lab.gold.obt_orders
GROUP BY category, Shipping_city, Shipping_state
ORDER BY total_revenue DESC;

In [0]:
SELECT
  item_id,
  item_name,
  ROUND(SUM(total_amount), 2) AS amount_sold
FROM de_lab.gold.obt_orders
GROUP BY item_id, item_name
ORDER BY amount_sold DESC
LIMIT 5;